<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 4: Implementing a GPT model from Scratch To Generate Text 

## 4.1 Coding an LLM architecture

In [23]:
# cfg
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [31]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
        *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [32]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
print(batch)
batch.append(torch.tensor(tokenizer.encode(txt2)))
print(batch)
batch = torch.stack(batch, dim=0)
print(batch)

[tensor([6109, 3626, 6100,  345])]
[tensor([6109, 3626, 6100,  345]), tensor([6109, 1110, 6622,  257])]
tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [35]:
batch.shape

torch.Size([2, 4])

In [41]:
# Initializing a new dummymodel using the GPT_CONFIG_124M values
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)
print(logits.shape)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 4, 50257])


## 4.2 Normalizing activations with layer normalization

In [49]:
torch.manual_seed(123)

batch_example = torch.randn(2, 5)
batch_example

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

In [95]:
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
out

tensor([[0.0000, 0.0000, 0.1032, 0.0000, 0.0000, 0.0000],
        [0.0038, 0.0000, 0.0000, 0.3069, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)

In [96]:
out.mean(dim=0) # Batch normalization, not recommended
out.mean(dim=-1) # Normalizes each layer

tensor([0.0172, 0.0518], grad_fn=<MeanBackward1>)

In [97]:
mean = out.mean(dim=-1, keepdim=True)
mean

tensor([[0.0172],
        [0.0518]], grad_fn=<MeanBackward1>)

In [104]:
var = out.var(dim=-1, keepdim=True)
var

tensor([[0.0018],
        [0.0156]], grad_fn=<VarBackward0>)

In [107]:
torch.set_printoptions(sci_mode=False) # Gives a clean zero value
(out - mean).mean(dim=-1, keepdim=True) # Subtract 'out' with mean gives zero

tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)

In [108]:
# Create a variance of 1
normed = ((out - mean) / torch.sqrt(var))
normed.var(dim=-1, keepdim=True)

tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)

In [119]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        # Parameter = trainable weights
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # Same as we calculated before
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True) #, unbiased=True)
        norm_x = (x - mean) / torch.sqrt(var + self.eps) # self.eps prevents dividing by zero error
        # self_shift is zero at first, but will undo .mean, 
        # and self.scale does the same to norm_x division
        # to iterate towards good weight values
        return self.scale * norm_x + self.shift

In [120]:
ln = LayerNorm(6)
outputs_normed = ln(out)

In [121]:
outputs_normed

tensor([[-0.4071, -0.4071,  2.0355, -0.4071, -0.4071, -0.4071],
        [-0.3839, -0.4141, -0.4141,  2.0404, -0.4141, -0.4141]],
       grad_fn=<AddBackward0>)

In [122]:
outputs_normed.mean(dim=-1, keepdim=True)

tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)

In [123]:
outputs_normed.var(dim=-1, keepdim=True)

tensor([[0.9944],
        [0.9994]], grad_fn=<VarBackward0>)

## 4.3 Implementing a feed forward network with GELU activations